In [2]:
import pandas as pd

df = pd.read_json('processed_df.jsonl', orient="records", lines=True)

len(df)

14455

In [6]:
df["approx_chars"] = df["maintext"].str.len()
print("Mean chars:", df["approx_chars"].mean())

Mean chars: 3952.5267381528884


In [3]:
import importlib
import relevance_prompts

importlib.reload(relevance_prompts)
PROMPT_TEXT = relevance_prompts.PROMPT

# Cell 1 — SETUP ONLY for OPENAI BATCH API (binary relevance, Variant A, GPT-5) — LABEL ONLY.
# - Builds Batch JSONL requests (one per row) with custom_id=row_id
# - No API calls are executed in this cell
# - Keeps gold-label normalization helpers for later evaluation

import os, re, json, time
from dataclasses import dataclass
from typing import Optional, Dict, Any, Iterable

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

PROMPT_SPEC_A = {
    "name": "relevance_binary_variantA",
    "system": "You are a careful, literal classifier. Follow the rules and output format exactly. Output ONLY valid JSON.",
    "user_template": PROMPT_TEXT,  # expects {title} and {text}
}

# ---- LABELS (binary) ----
LABELS = ["IRRELEVANT", "RELEVANT"]

def normalize_gold_binary(consensus_binary) -> str:
    v = int(consensus_binary)
    return "IRRELEVANT" if v == 0 else "RELEVANT"

def extract_first_json_object(text: str):
    if not text:
        return None
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None

def normalize_pred_binary(label) -> Optional[str]:
    if label is None:
        return None
    s = str(label).strip().lower()
    if "irre" in s or s in {"0", "false", "no"}:
        return "IRRELEVANT"
    if "rel" in s or s in {"1", "true", "yes"}:
        return "RELEVANT"
    return None

def _render_user_template(user_template: str, title: str, text: str) -> str:
    """
    Safe formatter:
    - Allows {title} and {text}
    - Escapes all other braces (so JSON examples in prompt don't break .format)
    """
    template = user_template.replace("{", "{{").replace("}", "}}")
    template = template.replace("{{title}}", "{title}").replace("{{text}}", "{text}")
    return template.format(title=title, text=text)

def _safe_str(x: Any) -> str:
    """Convert to a usable string; treat NaN/None as empty."""
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    s = str(x)
    if s.strip().lower() in {"nan", "none"}:
        return ""
    return s

@dataclass
class BatchConfig:
    model: str = "gpt-5"
    max_output_tokens: int = 100  # label-only: keep small
    reasoning_effort: str = "minimal"  # reduce reasoning token burn
    text_verbosity: str = "low"    # encourage short output
    endpoint: str = "/v1/responses"  # Batch API will target this endpoint

# ---- Batch builder ----
def build_batch_jsonl(
    df: pd.DataFrame,
    prompt_spec: Dict[str, str],
    out_jsonl_path: str,
    cfg: Optional[BatchConfig] = None,
    row_id_col: str = "row_id",
    title_col: str = "title",
    text_col: str = "maintext",
    start: int = 0,
    stop: Optional[int] = None,
    overwrite: bool = True,
) -> str:
    """
    Writes a JSONL file for OpenAI Batch API where each line is one request:
      {"custom_id": "...", "method": "POST", "url": "/v1/responses", "body": {...}}

    Requirements:
      - prompt_spec["system"] and prompt_spec["user_template"] exist
      - user_template includes {title} and {text}
      - df has columns title_col and text_col
    """
    if cfg is None:
        cfg = BatchConfig()
    if not overwrite and os.path.exists(out_jsonl_path):
        raise FileExistsError(f"{out_jsonl_path} exists and overwrite=False")

    os.makedirs(os.path.dirname(out_jsonl_path) or ".", exist_ok=True)

    df_work = df.copy()
    if row_id_col not in df_work.columns:
        # stable row ids so you can map results back
        df_work = df_work.reset_index(drop=False).rename(columns={"index": row_id_col})

    if stop is None:
        stop = len(df_work)

    n_written = 0
    with open(out_jsonl_path, "w", encoding="utf-8") as f:
        for _, row in df_work.iloc[start:stop].iterrows():
            rid = row[row_id_col]
            title = _safe_str(row.get(title_col, ""))
            text = _safe_str(row.get(text_col, ""))

            user_msg = _render_user_template(
                prompt_spec["user_template"],
                title=title,
                text=text,
            )

            # Batch line format (one request)
            line = {
                "custom_id": str(rid),
                "method": "POST",
                "url": cfg.endpoint,
                "body": {
                    "model": cfg.model,
                    "input": [
                        {"role": "system", "content": prompt_spec["system"]},
                        {"role": "user", "content": user_msg},
                    ],
                    "max_output_tokens": cfg.max_output_tokens,
                    "reasoning": {"effort": cfg.reasoning_effort},
                    "text": {"verbosity": cfg.text_verbosity},
                },
            }

            f.write(json.dumps(line, ensure_ascii=False) + "\n")
            n_written += 1

    print(f"Wrote {n_written} requests to {out_jsonl_path}")
    return out_jsonl_path

# ---- Setup client (for later batch submit/poll/download steps) ----
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ---- Ensure binary gold labels exist (no API calls) ----
if "gold_label" not in df.columns:
    if "consensus_binary" in df.columns:
        df["gold_label"] = df["consensus_binary"].apply(normalize_gold_binary)
    # If this is your full 15k run without gold, it's fine to skip.
    # But if you're expecting gold for evaluation here, raise an error:
    # else:
    #     raise ValueError("Need df['gold_label'] or df['consensus_binary'] to create it.")


CHUNKING

In [12]:
import os, json, math
from datetime import datetime

# Assumes these already exist from your setup cell:
# - df
# - client
# - PROMPT_SPEC_A
# - BatchConfig
# - build_batch_jsonl

# -------------------------
# CONFIG (edit these only)
# -------------------------
run_name = "relevance15k"
RUN_TS = "v1"

chunk_size = 550
CHUNK_INDEX = 9  # change each run

cfg = BatchConfig(model="gpt-5", max_output_tokens=100, reasoning_effort="minimal", text_verbosity="low")

# -------------------------
# PREP
# -------------------------
os.makedirs("batches", exist_ok=True)

# Ensure stable row_id once
if "row_id" not in df.columns:
    df = df.reset_index(drop=False).rename(columns={"index": "row_id"})

n = len(df)
n_chunks = math.ceil(n / chunk_size)

if not (0 <= CHUNK_INDEX < n_chunks):
    raise ValueError(f"CHUNK_INDEX must be in [0, {n_chunks-1}] but got {CHUNK_INDEX}")

start = CHUNK_INDEX * chunk_size
stop = min((CHUNK_INDEX + 1) * chunk_size, n)

# Only file we keep long-term:
index_path = f"batches/{run_name}_{RUN_TS}_ALL_batches.json"

# Temporary input JSONL (we'll delete after upload + batch creation)
jsonl_path = f"batches/_TMP_{run_name}_{RUN_TS}_chunk{CHUNK_INDEX:03d}_{start}_{stop}.jsonl"

# -------------------------
# Safety: prevent accidental resubmit of same (chunk_index,start,stop)
# -------------------------
if os.path.exists(index_path):
    with open(index_path, "r") as f:
        index = json.load(f)
else:
    index = []

for rec in index:
    if int(rec.get("chunk_index", -1)) == int(CHUNK_INDEX) and int(rec.get("start", -1)) == int(start) and int(rec.get("stop", -1)) == int(stop):
        raise FileExistsError(
            f"Chunk {CHUNK_INDEX} with rows [{start}:{stop}) already recorded in {index_path}.\n"
            f"Batch id: {rec.get('batch_id')}"
        )

print(f"Submitting chunk {CHUNK_INDEX+1}/{n_chunks}: rows [{start}:{stop}) (n={stop-start})")

# -------------------------
# 1) Build JSONL for this chunk (TEMP)
# -------------------------
build_batch_jsonl(
    df=df,
    prompt_spec=PROMPT_SPEC_A,
    out_jsonl_path=jsonl_path,
    cfg=cfg,
    start=start,
    stop=stop,
    overwrite=True,
)

# -------------------------
# 2) Upload JSONL
# -------------------------
with open(jsonl_path, "rb") as f:
    batch_file = client.files.create(file=f, purpose="batch")

# -------------------------
# 3) Create Batch
# -------------------------
batch = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/responses",
    completion_window="24h",
)

print("Batch created:", batch.id)

# -------------------------
# 4) Record ONLY the essentials in ALL_batches.json
# -------------------------
record = {
    "run_name": run_name,
    "run_ts": RUN_TS,
    "chunk_index": int(CHUNK_INDEX),
    "chunk_size": int(chunk_size),
    "start": int(start),
    "stop": int(stop),
    "n_rows_chunk": int(stop - start),
    "batch_id": batch.id,
    "input_file_id": batch_file.id,
    "created_at": getattr(batch, "created_at", None),
}

index.append(record)
index = sorted(index, key=lambda d: d["chunk_index"])

with open(index_path, "w") as f:
    json.dump(index, f, indent=2)

print("Updated run index:", index_path)

# -------------------------
# 5) Delete the temp JSONL (optional but recommended)
# -------------------------
try:
    os.remove(jsonl_path)
    print("Deleted temp input JSONL:", jsonl_path)
except FileNotFoundError:
    pass

print("You can disconnect now. Later: client.batches.retrieve('<batch_id>')")

Submitting chunk 10/27: rows [4950:5500) (n=550)
Wrote 550 requests to batches/_TMP_relevance15k_v1_chunk009_4950_5500.jsonl
Batch created: batch_69a48498f10c81909e1ecbc3fafdc2da
Updated run index: batches/relevance15k_v1_ALL_batches.json
Deleted temp input JSONL: batches/_TMP_relevance15k_v1_chunk009_4950_5500.jsonl
You can disconnect now. Later: client.batches.retrieve('<batch_id>')


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # only needed if you're using .env

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

batch_id = "batch_69a48498f10c81909e1ecbc3fafdc2da"
st = client.batches.retrieve(batch_id)

print("status:", st.status)
print("request_counts:", st.request_counts)
print("errors:", st.errors)
print("error_file_id:", getattr(st, "error_file_id", None))
print("output_file_id:", getattr(st, "output_file_id", None))

status: completed
request_counts: BatchRequestCounts(completed=550, failed=0, total=550)
errors: None
error_file_id: None
output_file_id: file-4iHKvJg7RnmPeuqqEAvtAU


In [ ]:
import os, json
import pandas as pd
from typing import Optional, Dict, Any, Iterable

BATCH_ID = "batch_69a48498f10c81909e1ecbc3fafdc2da"

# If you truly don't want to keep raw batch output, set this to None.
# (If you keep it, it's useful for debugging later.)
OUT_PATH = "batches/chunk9.jsonl"

# Final artifact you want (row_id,label only)
FINAL_CSV_PATH = "batches/chunk9_final_labels.csv"

def extract_first_json_object(text: str):
    if not text:
        return None
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None
    
def normalize_pred_binary(label) -> Optional[str]:
    if label is None:
        return None
    s = str(label).strip().lower()
    if "irre" in s or s in {"0", "false", "no"}:
        return "IRRELEVANT"
    if "rel" in s or s in {"1", "true", "yes"}:
        return "RELEVANT"
    return None

# ---- 1) retrieve batch ----
b = client.batches.retrieve(BATCH_ID)
print("status:", b.status)
if b.status != "completed":
    raise RuntimeError(f"Batch not completed (status={b.status}).")

output_file_id = b.output_file_id
if not output_file_id:
    raise RuntimeError("No output_file_id found on completed batch.")

print("output_file_id:", output_file_id)

# ---- 2) download output file (temporarily, then optionally delete) ----
content = client.files.content(output_file_id)
with open(OUT_PATH, "wb") as f:
    f.write(content.read())
print("saved raw batch output:", OUT_PATH)

def extract_output_text_from_batch_line(obj: dict) -> str | None:
    """
    Raw Batch output line format (Responses API):
      obj["response"]["body"]["output"] is a list containing items like:
        - {"type": "reasoning", ...}
        - {"type": "message", "content": [{"type":"output_text", "text":"..."}], ...}

    Returns assistant output text or None (e.g., if incomplete).
    """
    resp = obj.get("response") or {}
    body = resp.get("body") or {}
    output = body.get("output") or []

    if not isinstance(output, list):
        return None

    for item in output:
        if item.get("type") == "message":
            content_list = item.get("content") or []
            if not isinstance(content_list, list):
                continue
            for c in content_list:
                if isinstance(c, dict) and isinstance(c.get("text"), str):
                    return c["text"]
    return None

def load_chunk_results_fixed(path: str) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)

            rid = obj.get("custom_id")
            text_out = extract_output_text_from_batch_line(obj)

            parsed = extract_first_json_object(text_out) if text_out else None
            label = normalize_pred_binary(parsed.get("label")) if isinstance(parsed, dict) else None

            rows.append({"row_id": str(rid) if rid is not None else None, "label": label})

    return pd.DataFrame(rows)

# ---- 3) parse ----
preds = load_chunk_results_fixed(OUT_PATH)

print("\nLabel counts:")
print(preds["label"].value_counts(dropna=False))

# Hard fail if any missing labels (prevents silent N/As)
missing = preds["label"].isna().sum()
if missing:
    raise RuntimeError(f"Found {missing} missing labels (None/NaN). Consider rerun for those row_ids.")

print("\nRelevant share:")
print((preds["label"] == "RELEVANT").mean())

print("\nSample parsed rows:")
print(preds.head(5))

# ---- 4) write final labels CSV (row_id,label only) ----
final_labels = preds[["row_id", "label"]].copy()
final_labels = final_labels.dropna(subset=["row_id"]).drop_duplicates(subset=["row_id"], keep="last")
final_labels = final_labels.sort_values("row_id", key=lambda s: s.astype(int))

os.makedirs(os.path.dirname(FINAL_CSV_PATH) or ".", exist_ok=True)
final_labels.to_csv(FINAL_CSV_PATH, index=False)
print("\nSaved final labels CSV:", FINAL_CSV_PATH)

# ---- 5) optional: delete raw batch output to save space ----
# (Uncomment to delete)
os.remove(OUT_PATH)
print("Deleted raw batch output:", OUT_PATH)

status: completed
output_file_id: file-4iHKvJg7RnmPeuqqEAvtAU
saved raw batch output: batches/chunk8.jsonl

Label counts:
label
RELEVANT      370
IRRELEVANT    180
Name: count, dtype: int64

Relevant share:
0.6727272727272727

Sample parsed rows:
  row_id       label
0   4400    RELEVANT
1   4401    RELEVANT
2   4402  IRRELEVANT
3   4403    RELEVANT
4   4404  IRRELEVANT

Saved final labels CSV: batches/chunk8_final_labels.csv
Deleted raw batch output: batches/chunk8.jsonl


In [10]:
import pandas as pd

dt1 = pd.read_csv('batches/chunk1_final_labels.csv')
dt1['label'].value_counts()

label
RELEVANT      335
IRRELEVANT    215
Name: count, dtype: int64

DEBUGGING

In [5]:
import os, json
from datetime import datetime
from openai import OpenAI
from dotenv import load_dotenv

# Assumes these already exist from your earlier setup:
# - df
# - PROMPT_SPEC_A
# - _safe_str
# - _render_user_template
# - build_batch_jsonl (not strictly needed here)
# - extract_first_json_object
# - normalize_pred_binary
# - BatchConfig dataclass (we'll instantiate it)

# -------------------------
# CONFIG YOU SET
# -------------------------
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

run_name = "relevance15k_gpt5_varA"
RUN_TS = "relevance_full_15k_v1"     # can be any stable string; keep same run label
CHUNK_INDEX = 0                      # which chunk had the 99 missing?
COMPLETED_CHUNK_OUTPUT = "batches/chunk000_output.jsonl"  # path to the downloaded output jsonl

# Choose rerun model settings:
# If your API accepts reasoning_effort="minimal", use it + small max_output_tokens.
# If not, set reasoning_effort="low" and max_output_tokens=400 instead.
RERUN_CFG = BatchConfig(
    model="gpt-5",
    max_output_tokens=60,
    reasoning_effort="minimal",
    text_verbosity="low",
)


# Also rerun the small number of json_parse_failed rows?
RERUN_PARSE_FAILED_TOO = True

# -------------------------
# PREP: ensure stable row_id exists
# -------------------------
if "row_id" not in df.columns:
    df = df.reset_index(drop=False).rename(columns={"index": "row_id"})
df["row_id"] = df["row_id"].astype(str)

os.makedirs("batches", exist_ok=True)

# -------------------------
# 1) Identify rows to rerun from the completed chunk output
# -------------------------
def extract_any_text_from_output_item(item: dict):
    content_list = item.get("content")
    if isinstance(content_list, list):
        for c in content_list:
            if isinstance(c, dict) and isinstance(c.get("text"), str):
                return c["text"]
    for k in ("text", "output_text"):
        v = item.get(k)
        if isinstance(v, str) and v.strip():
            return v
    return None

def extract_output_text_from_batch_line(obj: dict) -> str | None:
    resp = obj.get("response") or {}
    body = resp.get("body") or {}
    output = body.get("output") or []
    if not isinstance(output, list):
        return None

    # prefer message
    for item in output:
        if isinstance(item, dict) and item.get("type") == "message":
            t = extract_any_text_from_output_item(item)
            if t:
                return t

    # fallback any text
    for item in output:
        if isinstance(item, dict):
            t = extract_any_text_from_output_item(item)
            if t:
                return t

    return None

rerun_ids = []
parse_failed_ids = []

with open(COMPLETED_CHUNK_OUTPUT, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        rid = str(obj.get("custom_id"))

        resp = obj.get("response") or {}
        body = resp.get("body") or {}

        # Case A: incomplete due to max_output_tokens (your main problem)
        if body.get("status") == "incomplete":
            inc = body.get("incomplete_details") or {}
            if inc.get("reason") == "max_output_tokens":
                rerun_ids.append(rid)
                continue

        # Case B: completed but JSON parse failed (optional)
        if RERUN_PARSE_FAILED_TOO:
            raw_text = extract_output_text_from_batch_line(obj)
            parsed = extract_first_json_object(raw_text) if raw_text else None
            label = normalize_pred_binary(parsed.get("label")) if isinstance(parsed, dict) else None
            if label is None:
                # only count as parse-failed if there WAS some text, but it didn't parse into label
                if raw_text is not None and (parsed is None or "label" not in (parsed or {})):
                    parse_failed_ids.append(rid)

rerun_ids = sorted(set(rerun_ids))
parse_failed_ids = sorted(set(parse_failed_ids))

if RERUN_PARSE_FAILED_TOO:
    ids_to_rerun = sorted(set(rerun_ids + parse_failed_ids), key=lambda x: int(x))
else:
    ids_to_rerun = sorted(set(rerun_ids), key=lambda x: int(x))

print("Incomplete(max_output_tokens) rows:", len(rerun_ids))
print("Parse-failed rows (optional):", len(parse_failed_ids))
print("Total rows to rerun:", len(ids_to_rerun))

if len(ids_to_rerun) == 0:
    raise RuntimeError("No rows identified for rerun. Check COMPLETED_CHUNK_OUTPUT path or logic.")

# -------------------------
# 2) Build rerun JSONL (only those row_ids)
# -------------------------
rerun_jsonl_path = f"batches/{run_name}_{RUN_TS}_chunk{CHUNK_INDEX:03d}_RERUN_missing.jsonl"

id_set = set(ids_to_rerun)
n_written = 0

with open(rerun_jsonl_path, "w", encoding="utf-8") as out:
    # iterate only over relevant df rows (filter first to speed up)
    df_sub = df[df["row_id"].isin(id_set)]

    for _, row in df_sub.iterrows():
        rid = str(row["row_id"])
        title = _safe_str(row.get("title", ""))
        text = _safe_str(row.get("maintext", ""))

        user_msg = _render_user_template(PROMPT_SPEC_A["user_template"], title=title, text=text)

        line = {
            "custom_id": rid,
            "method": "POST",
            "url": RERUN_CFG.endpoint,
            "body": {
                "model": RERUN_CFG.model,
                "input": [
                    {"role": "system", "content": PROMPT_SPEC_A["system"]},
                    {"role": "user", "content": user_msg},
                ],
                "max_output_tokens": RERUN_CFG.max_output_tokens,
                "reasoning": {"effort": RERUN_CFG.reasoning_effort},
                "text": {"verbosity": RERUN_CFG.text_verbosity},
            },
        }
        out.write(json.dumps(line, ensure_ascii=False) + "\n")
        n_written += 1

print(f"Wrote {n_written} rerun requests to {rerun_jsonl_path}")

# -------------------------
# 3) Upload rerun JSONL + create rerun batch
# -------------------------
with open(rerun_jsonl_path, "rb") as f:
    rerun_file = client.files.create(file=f, purpose="batch")

rerun_batch = client.batches.create(
    input_file_id=rerun_file.id,
    endpoint="/v1/responses",
    completion_window="24h",
)

print("Rerun batch created:", rerun_batch.id)

# -------------------------
# 4) Save rerun batch id + metadata
# -------------------------
rerun_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
batch_id_path = f"batches/{run_name}_{RUN_TS}_chunk{CHUNK_INDEX:03d}_RERUN_batch_id.txt"
meta_path = f"batches/{run_name}_{RUN_TS}_chunk{CHUNK_INDEX:03d}_RERUN_batch_meta.json"

with open(batch_id_path, "w") as f:
    f.write(rerun_batch.id)

meta = {
    "run_name": run_name,
    "run_ts": RUN_TS,
    "chunk_index": CHUNK_INDEX,
    "rerun_created_ts": rerun_ts,
    "rerun_batch_id": rerun_batch.id,
    "rerun_input_file_id": rerun_file.id,
    "rerun_jsonl_path": rerun_jsonl_path,
    "n_rows_rerun": n_written,
    "reason_for_rerun": {
        "incomplete_max_output_tokens": len(rerun_ids),
        "json_parse_failed_optional": len(parse_failed_ids),
    },
    "cfg": {
        "model": RERUN_CFG.model,
        "max_output_tokens": RERUN_CFG.max_output_tokens,
        "reasoning_effort": RERUN_CFG.reasoning_effort,
        "text_verbosity": RERUN_CFG.text_verbosity,
        "endpoint": RERUN_CFG.endpoint,
    },
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved rerun batch id to:", batch_id_path)
print("Saved rerun metadata to:", meta_path)
print("You can disconnect now. Later: client.batches.retrieve(rerun_batch.id)")

Incomplete(max_output_tokens) rows: 100
Parse-failed rows (optional): 0
Total rows to rerun: 100
Wrote 100 rerun requests to batches/relevance15k_gpt5_varA_relevance_full_15k_v1_chunk000_RERUN_missing.jsonl
Rerun batch created: batch_69a223c242648190a85868eb7799c0e0
Saved rerun batch id to: batches/relevance15k_gpt5_varA_relevance_full_15k_v1_chunk000_RERUN_batch_id.txt
Saved rerun metadata to: batches/relevance15k_gpt5_varA_relevance_full_15k_v1_chunk000_RERUN_batch_meta.json
You can disconnect now. Later: client.batches.retrieve(rerun_batch.id)


In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # only needed if you're using .env

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

batch_id = "batch_69a223c242648190a85868eb7799c0e0"
st = client.batches.retrieve(batch_id)

print("status:", st.status)
print("request_counts:", st.request_counts)
print("errors:", st.errors)
print("error_file_id:", getattr(st, "error_file_id", None))
print("output_file_id:", getattr(st, "output_file_id", None))

status: completed
request_counts: BatchRequestCounts(completed=100, failed=0, total=100)
errors: None
error_file_id: None
output_file_id: file-DutW4xJY8tAAEBemwnz1KV


In [2]:
BATCH_ID = "batch_69a223c242648190a85868eb7799c0e0"  # from your saved chunk000_batch_id.txt
OUT_PATH = "batches/chunk0rerun.jsonl"  # wherever you want to save it

# ---- 1) retrieve batch ----
b = client.batches.retrieve(BATCH_ID)
print("status:", b.status)
if b.status != "completed":
    raise RuntimeError(f"Batch not completed (status={b.status}).")

output_file_id = b.output_file_id
if not output_file_id:
    raise RuntimeError("No output_file_id found on completed batch.")

print("output_file_id:", output_file_id)

# ---- 2) download output file ----
content = client.files.content(output_file_id)
with open(OUT_PATH, "wb") as f:
    f.write(content.read())
print("saved:", OUT_PATH)

status: completed
output_file_id: file-DutW4xJY8tAAEBemwnz1KV
saved: batches/chunk0rerun.jsonl


Merging N/A 

In [10]:
import json
import pandas as pd

# --------- helpers to extract text + parse label ---------

def extract_output_text_from_raw_batch_line(obj: dict) -> str | None:
    """
    Raw Batch output line format:
      obj["response"]["body"]["output"] is a list of items.
      We want the assistant message text (the JSON with {"label": ...}).

    Returns text or None if no message text exists (e.g., incomplete outputs).
    """
    resp = obj.get("response") or {}
    body = resp.get("body") or {}

    output = body.get("output") or []
    if not isinstance(output, list):
        return None

    # Prefer explicit message item
    for item in output:
        if isinstance(item, dict) and item.get("type") == "message":
            content_list = item.get("content") or []
            if isinstance(content_list, list):
                for c in content_list:
                    if isinstance(c, dict) and isinstance(c.get("text"), str):
                        return c["text"]

    # Fallback: any content with a text field
    for item in output:
        if isinstance(item, dict):
            content_list = item.get("content")
            if isinstance(content_list, list):
                for c in content_list:
                    if isinstance(c, dict) and isinstance(c.get("text"), str):
                        return c["text"]
            # some shapes might put text directly
            if isinstance(item.get("text"), str):
                return item["text"]

    return None

def parse_raw_batch_jsonl_to_labels(path: str) -> pd.DataFrame:
    """
    Returns DataFrame with columns:
      row_id (string), label (RELEVANT/IRRELEVANT or None),
      plus a few diagnostic columns.
    """
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)

            row_id = str(obj.get("custom_id")) if obj.get("custom_id") is not None else None
            resp = obj.get("response") or {}
            body = resp.get("body") or {}

            raw_text = extract_output_text_from_raw_batch_line(obj)
            parsed = extract_first_json_object(raw_text) if raw_text else None
            label = normalize_pred_binary(parsed.get("label")) if isinstance(parsed, dict) else None

            rows.append({
                "row_id": row_id,
                "label": label,
                "body_status": body.get("status"),
                "incomplete_reason": (body.get("incomplete_details") or {}).get("reason"),
                "status_code": resp.get("status_code"),
            })

    df_labels = pd.DataFrame(rows)
    return df_labels

# --------- paths (yours) ---------
raw_chunk0_path = "batches/chunk000_output.jsonl"
raw_rerun_path  = "batches/chunk0rerun.jsonl"

# --------- 1) parse both raw files ---------
labels0 = parse_raw_batch_jsonl_to_labels(raw_chunk0_path)
labels_rerun = parse_raw_batch_jsonl_to_labels(raw_rerun_path)

print("Chunk0 label counts (pre-fill):")
print(labels0["label"].value_counts(dropna=False))
print("\nRerun label counts:")
print(labels_rerun["label"].value_counts(dropna=False))

# Keep only the columns we need from rerun (row_id + label)
labels_rerun = labels_rerun[["row_id", "label"]].rename(columns={"label": "label_rerun"})

# --------- 2) fill missing labels in chunk0 with rerun labels ---------
merged = labels0.merge(labels_rerun, on="row_id", how="left")
merged["label"] = merged["label"].fillna(merged["label_rerun"])
merged = merged.drop(columns=["label_rerun"])

# Sanity checks
print("\nAfter fill:")
print("Missing labels:", merged["label"].isna().sum())
print(merged["label"].value_counts(dropna=False))

# Optional: keep only final label table for downstream merging
final_labels = merged[["row_id", "label"]].copy()
final_labels = final_labels.sort_values("row_id", key=lambda s: s.astype(int))

# --------- 3) save clean outputs ---------
final_labels.to_json("chunk000_final_labels.jsonl", orient="records", lines=True)
final_labels.to_csv("chunk000_final_labels.csv", index=False)

print("\nSaved:")
print(" - chunk000_final_labels.jsonl")
print(" - chunk000_final_labels.csv")

Chunk0 label counts (pre-fill):
label
RELEVANT      318
IRRELEVANT    133
None           99
Name: count, dtype: int64

Rerun label counts:
label
RELEVANT      51
IRRELEVANT    49
Name: count, dtype: int64

After fill:
Missing labels: 0
label
RELEVANT      369
IRRELEVANT    181
Name: count, dtype: int64

Saved:
 - chunk000_final_labels.jsonl
 - chunk000_final_labels.csv
